In [1]:
import time

In [2]:
# --- 개념적 구현을 위한 보조 클래스 ---

class PagedMemoryManager:
    """KV 캐시를 위한 블록 단위 메모리를 관리하는 가상 관리자."""
    def __init__(self, total_blocks):
        # 사용 가능한 물리 메모리 블록의 인덱스를 리스트로 관리
        self.free_blocks = list(range(total_blocks))
        # 각 요청이 어떤 블록들을 사용 중인지 추적
        self.used_blocks = {}
        print(f"[MemManager] {total_blocks}개의 물리 블록으로 메모리 초기화.")

    def allocate_block(self):
        """사용 가능한 블록을 하나 할당합니다."""
        if not self.free_blocks:
            return None # Out of Memory (OOM)
        block_idx = self.free_blocks.pop(0)
        return block_idx

    def free_request_blocks(self, request):
        """요청이 완료되면 해당 요청이 사용하던 모든 블록을 해제합니다."""
        if request.id in self.used_blocks:
            blocks_to_free = self.used_blocks.pop(request.id)
            self.free_blocks.extend(blocks_to_free)
            # 정렬하여 관리 용이성을 높임 (개념적)
            self.free_blocks.sort()
            print(f"[MemManager] 요청 {request.id}의 블록 {blocks_to_free} 해제. (사용 가능: {self.free_blocks})")

class Request:
    """하나의 사용자 요청을 표현하는 클래스."""
    def __init__(self, req_id, prompt_len, target_len):
        self.id = req_id
        self.prompt_len = prompt_len
        self.target_len = target_len
        self.current_len = prompt_len
        self.block_table = [] # 이 요청의 논리적 시퀀스에 매핑된 물리 블록들의 리스트

# --- 연속 배치 스케줄러 ---

class ContinuousBatchScheduler:
    def __init__(self, memory_manager):
        self.memory_manager = memory_manager
        self.running_requests = []
        self.waiting_queue = []

    def add_request(self, request):
        """새로운 요청을 대기 큐에 추가합니다."""
        self.waiting_queue.append(request)
        print(f"[Scheduler] >> 새로운 요청 {request.id}({request.target_len} 토큰) 대기 큐에 추가.")

    def schedule_step(self):
        """스케줄러의 메인 로직: 한 번의 생성 스텝을 실행합니다."""
        # 1. 대기 큐의 요청을 실행 큐로 이동 (메모리 할당 시도)
        # 실행 가능한 요청들을 담을 임시 리스트
        newly_admitted = []
        for req in self.waiting_queue:
            # 초기 KV 캐시를 저장할 블록을 할당받을 수 있는지 확인
            initial_block = self.memory_manager.allocate_block()
            if initial_block:
                req.block_table.append(initial_block)
                self.running_requests.append(req)
                self.memory_manager.used_blocks[req.id] = req.block_table
                newly_admitted.append(req)
                print(f"[Scheduler] 요청 {req.id} 시작. 초기 블록 {initial_block} 할당.")
            else:
                print(f"[Scheduler] 메모리 부족! 요청 {req.id}는 계속 대기합니다.")

        # 대기 큐에서 실행 큐로 옮겨진 요청들은 제거
        self.waiting_queue = [req for req in self.waiting_queue if req not in newly_admitted]

        if not self.running_requests:
            return False # 처리할 요청이 없으면 종료

        # 2. 패딩 없는 배치 구성
        # 실제로는 각 요청의 마지막 히든 상태 등을 모아 텐서를 만듭니다.
        # 여기서는 단순히 현재 실행 중인 요청들의 ID 리스트로 배치를 표현합니다.
        batch_to_run = [req.id for req in self.running_requests]
        print(f"\n--- 생성 스텝 실행 ---")
        print(f"  - 대상 배치 (패딩 없음): {batch_to_run}")

        # 3. GPU 연산 시뮬레이션 및 KV 캐시 업데이트
        completed_requests = []
        for req in self.running_requests:
            req.current_len += 1
            # 개념적으로 5 토큰마다 새 블록이 필요하다고 가정
            if (req.current_len - req.prompt_len) > 0 and (req.current_len - req.prompt_len) % 5 == 0:
                new_block = self.memory_manager.allocate_block()
                if new_block:
                    req.block_table.append(new_block)
                    print(f"  - 요청 {req.id}에 새 블록 {new_block} 할당됨. (현재 블록 테이블: {req.block_table})")
                else:
                    print(f"  - [OOM] 요청 {req.id}에 메모리 부족 발생! 요청을 중단합니다.")
                    completed_requests.append(req) # OOM으로 종료 처리

            if req.current_len >= req.target_len:
                completed_requests.append(req)

        # 4. 완료된 요청 처리 (메모리 해제)
        if completed_requests:
            for req in list(completed_requests): # 복사본으로 순회
                print(f"[Scheduler] << 요청 {req.id} 완료. (총 {req.current_len} 토큰 생성)")
                self.memory_manager.free_request_blocks(req)
                if req in self.running_requests:
                    self.running_requests.remove(req)

        return True

In [3]:
# --- 시뮬레이션 실행 ---
mem_manager = PagedMemoryManager(total_blocks=10)
scheduler = ContinuousBatchScheduler(mem_manager)

scheduler.add_request(Request(req_id='A', prompt_len=10, target_len=18))
scheduler.add_request(Request(req_id='B', prompt_len=5, target_len=12))

print("\n[INFO] --- 5 스텝 실행 시작 ---\n")
for i in range(5):
    scheduler.schedule_step()

print("\n[INFO] --- 중간에 새로운 요청 C 추가 ---\n")
scheduler.add_request(Request(req_id='C', prompt_len=8, target_len=15))

print("\n[INFO] --- 모든 요청 완료까지 실행 ---\n")
while scheduler.running_requests or scheduler.waiting_queue:
    if not scheduler.schedule_step():
        break
    time.sleep(0.1)

print("\n--- 모든 요청 처리 완료 ---")

[MemManager] 10개의 물리 블록으로 메모리 초기화.
[Scheduler] >> 새로운 요청 A(18 토큰) 대기 큐에 추가.
[Scheduler] >> 새로운 요청 B(12 토큰) 대기 큐에 추가.

[INFO] --- 5 스텝 실행 시작 ---

[Scheduler] 메모리 부족! 요청 A는 계속 대기합니다.
[Scheduler] 요청 B 시작. 초기 블록 1 할당.

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['B']
[Scheduler] 요청 A 시작. 초기 블록 2 할당.

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['B', 'A']

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['B', 'A']

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['B', 'A']

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['B', 'A']
  - 요청 B에 새 블록 3 할당됨. (현재 블록 테이블: [1, 3])

[INFO] --- 중간에 새로운 요청 C 추가 ---

[Scheduler] >> 새로운 요청 C(15 토큰) 대기 큐에 추가.

[INFO] --- 모든 요청 완료까지 실행 ---

[Scheduler] 요청 C 시작. 초기 블록 4 할당.

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['B', 'A', 'C']
  - 요청 A에 새 블록 5 할당됨. (현재 블록 테이블: [2, 5])

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['B', 'A', 'C']
[Scheduler] << 요청 B 완료. (총 12 토큰 생성)
[MemManager] 요청 B의 블록 [1, 3] 해제. (사용 가능: [1, 3, 6, 7, 8, 9])

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['A', 'C']

--- 생성 스텝 실행 ---
  - 대상 배치 (패딩 없음): ['A', '